# Tsetlin bake-off on a free Colab GPU

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
Stay on the tab — free Colab disconnects after ~90 min idle. Run is ~5–7 min.

Every step runs as a subprocess so the notebook kernel's own (numpy 2.x) state
can't interfere with our `numpy<2` install.

In [ ]:
# 1. GPU check + setup
!nvidia-smi -L || echo 'NO GPU — Runtime > Change runtime type > T4 GPU'
import os
if not os.path.isdir('tsetlin-market-lab'):
    !git clone --depth 1 https://github.com/naibwedi/tsetlin-market-lab.git
os.chdir('/content/tsetlin-market-lab')
!pip -q install 'numpy<2' 'scikit-learn==1.5.2' pandas pyarrow pyyaml python-dotenv xgboost lightgbm tmu pycuda 2>&1 | tail -1
print('cwd', os.getcwd())

In [ ]:
# 2. Features (synthetic until real odds are collected) + the 7 baselines
import glob
if not glob.glob('data/features/X.parquet'):
    !python -m src.ingest.make_synthetic --n-matches 80
    !python -m src.panel.build_panel --config config/features.yaml
    !python -m src.features.booleanize --config config/features.yaml
!python -m src.models.bakeoff --config config/bakeoff.ci.yaml
print('\n' + open('results/summary.md').read())

In [ ]:
# 3. Train the Tsetlin Machine on the GPU (subprocess -> clean numpy)
!python -m scripts.tm_run

In [ ]:
# 4. The result + the rules it learned
print(open('results/tm_result.json').read())
print('\n--- clauses ---')
print(open('results/tm_clauses.txt').read())